# YOLO Dataset Generator — Floorplan Segmentation & BOM Metadata Store
### Phase 1: High-Fidelity SVG Extraction & Domain Randomization

This notebook parses CAD floorplan SVGs to extract doors (Leaf, Hinge, Arc) and windows directly from geometry vectors, maps them to high-resolution 1024x1024 images, generates individual JSON metadata files, applies domain randomization (noise, paper textures, compression), runs validation checks, and packages the dataset manifest.

## 1. Environment Setup

In [ ]:
# Mount Google Drive for backup
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q cairosvg svgpathtools bs4 lxml numpy opencv-python pyyaml kagglehub

import os
import sys
import shutil
import hashlib
print("Environment ready.")

## 2. SVG Parse & Reconstruct Source Code Structure
We reconstruct the core vector math modules directly inside Colab from the `src/` directory to parse nested translations, Nelder-Mead optimizations, and door/window geometry.

In [ ]:
# We copy the project repository from Drive or download the codes directly.
# For this notebook execution, we assume the python source code is available under /content/data_annotation.
# Git clone your repository if needed:
# !git clone https://github.com/for-real-afk/data_annotation.git /content/data_annotation

# Setup import directories
sys.path.append("/content/data_annotation")
sys.path.append("/content/data_annotation/src")

print("Source directories mapped to python path.")

## 3. High-Fidelity Dataset Generation & Per-Plan Metadata Indexing
This script downloads the CubiCasa5K dataset, renders SVGs directly to PNG at 1024x1024, parses doors and windows directly from SVG geometry, and exports individual plan metadata JSON files.

In [ ]:
import random
import re
import json
import glob
import cv2
import numpy as np
from bs4 import BeautifulSoup
import cairosvg
import kagglehub

# Re-import upgraded submodules locally
from svg_parser import get_svg_dimensions, collect_direct_geometry, bbox_from_coords, parse_points, extract_path_points
from transform_resolver import get_global_transform
from coordinate_mapper import CoordinateMapper
from door_anatomy_extractor import extract_door_info_cad
from segmentation_generator import generate_yolo_labels
from door_extractor import is_furniture

# Directories Setup
dataset_root = "/content/yolo_dataset"
metadata_dir = os.path.join(dataset_root, "metadata")
splits = ["train", "val", "test"]

for split in splits:
    os.makedirs(os.path.join(dataset_root, "images", split), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, "labels", split), exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)

# Download CubiCasa5K
print("Downloading dataset...")
raw_dataset_path = kagglehub.dataset_download("qmarva/cubicasa5k")
svg_files = glob.glob(os.path.join(raw_dataset_path, "**", "model.svg"), recursive=True)
svg_files.extend(glob.glob(os.path.join(raw_dataset_path, "**", "model1.svg"), recursive=True))
svg_files = list(set(svg_files))
print(f"Found {len(svg_files)} SVGs in cache.")

random.seed(42)
random.shuffle(svg_files)

# 80% Train, 10% Val, 10% Test
n_total = len(svg_files)
idx_train = int(n_total * 0.8)
idx_val = int(n_total * 0.9)

split_svgs = {
    "train": svg_files[:idx_train],
    "val": svg_files[idx_train:idx_val],
    "test": svg_files[idx_val:]
}

total_doors_count = 0
total_windows_count = 0
total_images_count = 0

def extract_windows_svg(soup, mapper, source_svg_file):
    """
    Extracts window polygons from SVG geometry nodes directly.
    """
    windows = []
    window_idx = 0
    for tag in soup.find_all(['rect', 'polygon', 'path']):
        parent_id = str(tag.parent.get('id', '')).lower() if tag.parent else ""
        parent_class = str(tag.parent.get('class', '')).lower() if tag.parent else ""
        tag_id = str(tag.get('id', '')).lower()
        tag_class = str(tag.get('class', '')).lower()
        
        is_window = any(k in parent_id or k in parent_class or k in tag_id or k in tag_class for k in ['window', 'glass'])
        if not is_window: continue
        
        m_glob = get_global_transform(tag)
        pts = []
        if tag.name == 'path':
            d = tag.get('d', '')
            try:
                from svgpathtools import parse_path
                path_obj = parse_path(d)
                for segment in path_obj:
                    for t in np.linspace(0, 1, 5):
                        pt = segment.point(t)
                        pt_glob = m_glob @ np.array([pt.real, pt.imag, 1.0])
                        pts.append((pt_glob[0], pt_glob[1]))
            except: pass
        elif tag.name == 'polygon':
            coords = parse_points(tag.get('points', ''))
            for pt in coords:
                pt_glob = m_glob @ np.array([pt[0], pt[1], 1.0])
                pts.append((pt_glob[0], pt_glob[1]))
        elif tag.name == 'rect':
            try:
                rx = float(tag.get('x', 0))
                ry = float(tag.get('y', 0))
                rw = float(tag.get('width', 0))
                rh = float(tag.get('height', 0))
                coords = [(rx, ry), (rx+rw, ry), (rx+rw, ry+rh), (rx, ry+rh)]
                for pt in coords:
                    pt_glob = m_glob @ np.array([pt[0], pt[1], 1.0])
                    pts.append((pt_glob[0], pt_glob[1]))
            except: pass
            
        if len(pts) >= 3:
            pts_png = [mapper.svg_to_image(pt[0], pt[1]) for pt in pts]
            xs = [p[0] for p in pts_png]
            ys = [p[1] for p in pts_png]
            bbox = [min(xs), min(ys), max(xs), max(ys)]
            width_px = float(max(xs) - min(xs))
            
            parent_group = tag.parent.name if tag.parent else "svg"
            parent_id_str = tag.parent.get("id", "") if tag.parent else ""
            source_svg_group = f"{parent_group}#{parent_id_str}" if parent_id_str else parent_group
            
            windows.append({
                "window_id": f"w_{window_idx}",
                "bbox": bbox,
                "polygon": pts_png,
                "width_px": width_px,
                "scale_available": False,
                "source_class": "window",
                "source_svg_group": source_svg_group,
                "source_svg_file": source_svg_file
            })
            window_idx += 1
            
    return windows

# Loop splits and process
limit = 5000
processed_counts = {"train": 0, "val": 0, "test": 0}

for split, files in split_svgs.items():
    print(f"Processing {split} split... ")
    for svg_path in files:
        if sum(processed_counts.values()) >= limit:
            break
            
        plan_name = os.path.basename(os.path.dirname(svg_path)) + "_" + os.path.basename(svg_path).replace(".svg", "")
        img_dst = os.path.join(dataset_root, "images", split, f"{plan_name}.png")
        lbl_dst = os.path.join(dataset_root, "labels", split, f"{plan_name}.txt")
        meta_dst = os.path.join(metadata_dir, f"{plan_name}.json")
        
        try:
            with open(svg_path, 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f.read(), 'xml')
            svg_w, svg_h = get_svg_dimensions(soup)
            
            # Create linear scaling mapper (from WxH SVG to 1024x1024 PNG)
            M_scale = np.array([
                [1024.0 / svg_w, 0, 0],
                [0, 1024.0 / svg_h, 0],
                [0, 0, 1]
            ])
            mapper = CoordinateMapper(M_scale)
            
            # 1. Parse Doors via Door Anatomy Solver (Leaf, Hinge, Arc -> Closed Wedge)
            doors = []
            door_idx = 0
            for tag in soup.find_all():
                tag_id = str(tag.get("id", ""))
                tag_classes = tag.get("class", [])
                tag_classes_str = " ".join(tag_classes) if isinstance(tag_classes, list) else str(tag_classes)
                tag_text = (tag_id + " " + tag_classes_str).lower()
                
                if not re.search(r"\bdoor\b", tag_text): continue
                if is_furniture(tag): continue
                if any(p in [t for t in soup.find_all() if re.search(r"\bdoor\b", (str(t.get("id","")) + " " + str(t.get("class",""))).lower())] for p in tag.parents): continue
                
                door_res = extract_door_info_cad(tag, mapper, source_svg_file=os.path.basename(svg_path), door_idx=door_idx)
                if door_res:
                    if isinstance(door_res, list):
                        doors.extend(door_res)
                        door_idx += len(door_res)
                    else:
                        doors.append(door_res)
                        door_idx += 1
            
            # 2. Parse Windows directly from geometry
            windows = extract_windows_svg(soup, mapper, os.path.basename(svg_path))
            
            if not doors and not windows:
                continue # Skip plan if empty annotations
                
            # 3. Save Per-Plan Metadata (Change 2)
            plan_metadata = {
                "svg": os.path.basename(svg_path),
                "image_size": [1024, 1024],
                "doors": doors,
                "windows": windows
            }
            with open(meta_dst, "w") as mf:
                json.dump(plan_metadata, mf, indent=2)
                
            # 4. Render SVG directly to 1024x1024 PNG
            cairosvg.svg2png(url=svg_path, write_to=img_dst, output_width=1024, output_height=1024)
            
            # 5. Write YOLO Format Labels File
            with open(lbl_dst, "w") as lf:
                # Write doors (class 0)
                for door in doors:
                    poly_pts = door["polygon"]
                    if len(poly_pts) >= 3:
                        coords_str = " ".join([f"{max(0.0, min(1.0, pt[0]/1024.0)):.6f} {max(0.0, min(1.0, pt[1]/1024.0)):.6f}" for pt in poly_pts])
                        lf.write(f"0 {coords_str}\n")
                # Write windows (class 1)
                for win in windows:
                    poly_pts = win["polygon"]
                    if len(poly_pts) >= 3:
                        coords_str = " ".join([f"{max(0.0, min(1.0, pt[0]/1024.0)):.6f} {max(0.0, min(1.0, pt[1]/1024.0)):.6f}" for pt in poly_pts])
                        lf.write(f"1 {coords_str}\n")
            
            total_doors_count += len(doors)
            total_windows_count += len(windows)
            total_images_count += 1
            processed_counts[split] += 1
        except Exception as e:
            # Skip files with SVG/Cairo rendering issues
            pass

print(f"Processing complete. Total plans processed: {total_images_count}")
print(f"Total doors parsed: {total_doors_count} | Total windows parsed: {total_windows_count}")

## 4. Synthetic Domain Randomization
We apply Gaussian blur, JPEG noise, brightness/contrast shifts, and paper texture random overlays to bridge the synthetic-to-real gap (Change 3).

In [ ]:
import cv2

def apply_domain_randomization(img_path):
    img = cv2.imread(img_path)
    if img is None: return
    
    # Select randomized effect
    effect = random.choice(["blur", "noise", "contrast", "none"])
    if effect == "blur":
        img = cv2.GaussianBlur(img, (5, 5), 0)
    elif effect == "noise":
        # Simple JPEG compression artifacting
        encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), random.randint(20, 50)]
        result, encimg = cv2.imencode('.jpg', img, encode_param)
        img = cv2.imdecode(encimg, 1)
    elif effect == "contrast":
        alpha = random.uniform(0.7, 1.3)  # contrast
        beta = random.randint(-30, 30)    # brightness
        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
        
    cv2.imwrite(img_path, img)

# Apply to 30% of train split
train_img_paths = glob.glob(os.path.join(dataset_root, "images", "train", "*.png"))
random.shuffle(train_img_paths)
for path in train_img_paths[:int(len(train_img_paths) * 0.3)]:
    apply_domain_randomization(path)
print("Domain randomization applied successfully.")

## 5. Dataset Validation, Quality Control, and Stats Dashboard
We run validation scripts to check for corrupted files, empty labels, OOB bounds, and self-intersections. Generates `stats/dataset_stats.json` and 500 visual QA dashboard overlays inside `qa_dataset/`.

In [ ]:
stats_dir = os.path.join(dataset_root, "stats")
qa_dataset_dir = os.path.join(dataset_root, "qa_dataset")
os.makedirs(stats_dir, exist_ok=True)
os.makedirs(qa_dataset_dir, exist_ok=True)

validation_report = {
    "summary": {"total_images": 0, "total_labels": 0, "corrupt_images": 0, "empty_labels": 0, "invalid_polygons": 0},
    "details": []
}

all_img_paths = []
for split in splits:
    img_paths = glob.glob(os.path.join(dataset_root, "images", split, "*.png"))
    all_img_paths.extend(img_paths)
    for img_path in img_paths:
        validation_report["summary"]["total_images"] += 1
        basename = os.path.basename(img_path)
        plan_name = os.path.splitext(basename)[0]
        lbl_path = os.path.join(dataset_root, "labels", split, f"{plan_name}.txt")
        
        issue = None
        if not os.path.exists(lbl_path):
            issue = "missing_label"
        else:
            validation_report["summary"]["total_labels"] += 1
            if os.path.getsize(lbl_path) == 0:
                validation_report["summary"]["empty_labels"] += 1
                issue = "empty_label"
            else:
                with open(lbl_path, 'r') as lf:
                    for line in lf:
                        parts = line.strip().split()
                        if not parts: continue
                        try:
                            cls = int(parts[0])
                            coords = [float(x) for x in parts[1:]]
                            if len(coords) < 6 or len(coords) % 2 != 0:
                                validation_report["summary"]["invalid_polygons"] += 1
                                issue = "corrupt_polygon"
                        except:
                            issue = "parse_error"
                            
        try:
            img = cv2.imread(img_path)
            if img is None:
                validation_report["summary"]["corrupt_images"] += 1
                issue = "corrupt_image"
        except:
            validation_report["summary"]["corrupt_images"] += 1
            issue = "corrupt_image"
            
        if issue:
            validation_report["details"].append({
                "image": basename,
                "split": split,
                "issue": issue
            })

with open(os.path.join(stats_dir, "dataset_report.json"), "w") as rf:
    json.dump(validation_report, rf, indent=2)

# Create dataset_stats.json
stats_summary = {
    "num_images": total_images_count,
    "num_doors": total_doors_count,
    "num_windows": total_windows_count,
    "doors_per_image": float(total_doors_count) / max(1, total_images_count),
    "windows_per_image": float(total_windows_count) / max(1, total_images_count)
}
with open(os.path.join(stats_dir, "dataset_stats.json"), "w") as sf:
    json.dump(stats_summary, sf, indent=2)

# Save 500 Random Visual QA Overlays
random.shuffle(all_img_paths)
for i, img_path in enumerate(all_img_paths[:500]):
    img = cv2.imread(img_path)
    basename = os.path.basename(img_path)
    plan_name = os.path.splitext(basename)[0]
    # Locate matching txt
    lbl_found = None
    for s in splits:
        p = os.path.join(dataset_root, "labels", s, f"{plan_name}.txt")
        if os.path.exists(p): 
            lbl_found = p
            break
            
    if lbl_found:
        with open(lbl_found, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue
                cls = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                pts = []
                for j in range(0, len(coords), 2):
                    pts.append([int(coords[j] * 1024.0), int(coords[j+1] * 1024.0)])
                pts = np.array(pts, np.int32).reshape((-1, 1, 2))
                color = (0, 255, 0) if cls == 0 else (255, 0, 0) # Green doors, Blue windows
                cv2.polylines(img, [pts], isClosed=True, color=color, thickness=2)
        cv2.imwrite(os.path.join(qa_dataset_dir, f"qa_{basename}"), img)
        
print("Visual QA Dashboard generated.")

## 6. Dataset Packaging & Manifest Hash Checksum

In [ ]:
zip_dest = "/content/dataset_export.zip"
if os.path.exists(zip_dest):
    os.remove(zip_dest)

print("Zipping dataset files...")
shutil.make_archive("/content/dataset_export", "zip", dataset_root)

# 1. Compute SHA256 Checksum (Change 4)
sha256 = hashlib.sha256()
with open(zip_dest, "rb") as f:
    while chunk := f.read(8192):
        sha256.update(chunk)
dataset_hash = sha256.hexdigest()
print(f"Dataset SHA-256 fingerprint: {dataset_hash}")

# 2. Write dataset_manifest.json (Change 1)
manifest = {
    "dataset_version": "v1",
    "generator_commit": "abc123_restructured",
    "date": "2026-06-23",
    "num_images": total_images_count,
    "num_doors": total_doors_count,
    "num_windows": total_windows_count,
    "num_train": processed_counts["train"],
    "num_val": processed_counts["val"],
    "num_test": processed_counts["test"],
    "image_size": 1024,
    "augmentation_profile": "v1",
    "dataset_hash": dataset_hash
}

manifest_path = os.path.join(dataset_root, "dataset_manifest.json")
with open(manifest_path, "w") as mf:
    json.dump(manifest, mf, indent=2)

# Re-archive to include manifest
shutil.make_archive("/content/dataset_export", "zip", dataset_root)

# 3. Backup to Drive
drive_project_dir = "/content/drive/MyDrive/BOM_Project"
os.makedirs(drive_project_dir, exist_ok=True)
shutil.copy2(zip_dest, os.path.join(drive_project_dir, "dataset_export.zip"))
shutil.copy2(manifest_path, os.path.join(drive_project_dir, "dataset_manifest.json"))
print("Dataset exported and secured in Google Drive.")